In [1]:
import asyncio
from decimal import Decimal
from datetime import datetime, timedelta, timezone
from typing_extensions import Literal

from typed_kucoin import KuCoin
from typed_kucoin.schemas import (
  FuturesAddOrderLimit,
  FuturesAddOrderMarket,
  FuturesOrderBookLevelUpdate,
  FuturesOrderEvent,
  HfAddOrderLimit,
  HfAddOrderMarket,
)
from typed_kucoin.streams.spot_margin_private.order_v2 import OrderChangeEvent
from typed_kucoin.streams.spot_margin_public.orderbook_level5 import (
  OrderbookLevel5Update,
)
from dotenv import load_dotenv

from tribulnation.sdk.market import (
  Book,
  Collateral,
  PerpCollateral,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpPosition,
  Position,
  Rules,
  Settings,
  Trade,
)

load_dotenv()

client = await KuCoin.new().__aenter__()

MARKETS = {
  'spot': ['BTC-USDT', 'ETH-USDT', 'KCS-USDT'],
  'perp': ['XBTUSDTM', 'ETHUSDTM', 'KCSUSDTM'],
}

## `Market` (spot)

In [2]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  size = '100' if levels and levels > 20 else '20'
  raw = await client.spot.part_orderbook(size, symbol=symbol)
  book = Book(
    bids=[Book.Entry(Decimal(p), Decimal(q)) for p, q in raw['bids']],
    asks=[Book.Entry(Decimal(p), Decimal(q)) for p, q in raw['asks']],
  )
  return book.limit(levels) if levels else book


{symbol: await depth(symbol, levels=5) for symbol in MARKETS['spot']}

{'BTC-USDT': Book(bids=[Book.Entry(price=Decimal('79681.3'), qty=Decimal('0.61211366')), Book.Entry(price=Decimal('79681.2'), qty=Decimal('0.01299999')), Book.Entry(price=Decimal('79681.1'), qty=Decimal('0.003')), Book.Entry(price=Decimal('79681'), qty=Decimal('0.08955518')), Book.Entry(price=Decimal('79680.9'), qty=Decimal('0.56484537'))], asks=[Book.Entry(price=Decimal('79681.4'), qty=Decimal('0.00371992')), Book.Entry(price=Decimal('79681.5'), qty=Decimal('0.003')), Book.Entry(price=Decimal('79681.6'), qty=Decimal('0.0275')), Book.Entry(price=Decimal('79681.7'), qty=Decimal('0.0095')), Book.Entry(price=Decimal('79682'), qty=Decimal('0.0645'))]),
 'ETH-USDT': Book(bids=[Book.Entry(price=Decimal('2458.16'), qty=Decimal('0.7928736')), Book.Entry(price=Decimal('2458.08'), qty=Decimal('3.0308244')), Book.Entry(price=Decimal('2458.05'), qty=Decimal('1.5601709')), Book.Entry(price=Decimal('2458.03'), qty=Decimal('2.0930478')), Book.Entry(price=Decimal('2458.02'), qty=Decimal('2.3473059'))]

In [3]:
def depth_stream(symbol: str):
  def to_book(update: OrderbookLevel5Update) -> Book:
    return Book(
      bids=[Book.Entry(p, q) for p, q in update['bids']],
      asks=[Book.Entry(p, q) for p, q in update['asks']],
    )

  return client.streams.spot_margin_public.orderbook_level5(symbol).map(to_book)


books: list[Book] = []
async with depth_stream(MARKETS['spot'][0]) as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('79681.3'), qty=Decimal('0.57451724')), Book.Entry(price=Decimal('79681.2'), qty=Decimal('0.01299999')), Book.Entry(price=Decimal('79681.1'), qty=Decimal('0.003')), Book.Entry(price=Decimal('79681'), qty=Decimal('0.08955518')), Book.Entry(price=Decimal('79680.9'), qty=Decimal('0.60238338'))], asks=[Book.Entry(price=Decimal('79681.4'), qty=Decimal('0.00371992')), Book.Entry(price=Decimal('79681.5'), qty=Decimal('0.003')), Book.Entry(price=Decimal('79681.6'), qty=Decimal('0.0275')), Book.Entry(price=Decimal('79681.7'), qty=Decimal('0.0095')), Book.Entry(price=Decimal('79682'), qty=Decimal('0.0645'))]),
 Book(bids=[Book.Entry(price=Decimal('79681.3'), qty=Decimal('0.57451724')), Book.Entry(price=Decimal('79681.2'), qty=Decimal('0.01299999')), Book.Entry(price=Decimal('79681.1'), qty=Decimal('0.003')), Book.Entry(price=Decimal('79681'), qty=Decimal('0.08955518')), Book.Entry(price=Decimal('79680.9'), qty=Decimal('0.60238338'))], asks=[Book.Entry(price=D

In [4]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  sym, fees = await asyncio.gather(
    client.spot.symbol(symbol),
    client.account.trade_fee.actual_fee(symbols=symbol),
  )
  fee = fees[0]
  return Rules(
    base=sym['baseCurrency'],
    quote=sym['quoteCurrency'],
    fee_asset=sym['feeCurrency'],
    tick_size=Decimal(sym['priceIncrement']),
    step_size=Decimal(sym['baseIncrement']),
    fixed_min_qty=Decimal(sym['baseMinSize']),
    min_value=Decimal(sym['quoteMinSize']) if sym.get('quoteMinSize') else None,
    max_qty=Decimal(sym['baseMaxSize']) if sym.get('baseMaxSize') else None,
    maker_fee=Decimal(fee['makerFeeRate']),
    taker_fee=Decimal(fee['takerFeeRate']),
    api=sym['enableTrading'],
    details=sym,
  )


{symbol: await rules(symbol) for symbol in MARKETS['spot']}

{'BTC-USDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('1E-8'), fixed_min_qty=Decimal('0.00001'), min_value=Decimal('0.1'), max_qty=Decimal('10000000000'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.001'), taker_fee=Decimal('0.001'), api=True, details={'symbol': 'BTC-USDT', 'name': 'BTC-USDT', 'baseCurrency': 'BTC', 'quoteCurrency': 'USDT', 'feeCurrency': 'USDT', 'market': 'USDS', 'baseMinSize': Decimal('0.00001'), 'baseMaxSize': Decimal('10000000000'), 'quoteMinSize': Decimal('0.1'), 'quoteMaxSize': Decimal('99999999'), 'baseIncrement': Decimal('1E-8'), 'quoteIncrement': Decimal('0.000001'), 'priceIncrement': Decimal('0.1'), 'priceLimitRate': Decimal('0.01'), 'minFunds': Decimal('0.1'), 'isMarginEnabled': True, 'enableTrading': True, 'feeCategory': 1, 'makerFeeCoefficient': Decimal('1.00'), 'takerFeeCoefficient': Decimal('1.00'), 'st': False, 'callauctionIsEnabled': False,

In [5]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.spot.orders_hf.get_open_orders(symbol=symbol)
  out: list[OrderState] = []
  for o in raw or []:
    size = Decimal(o['size'])
    dealt = Decimal(o['dealSize'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(
      OrderState(
        id=o['id'],
        price=Decimal(o['price']),
        qty=sign * size,
        filled_qty=sign * dealt,
        active=True,
        details=o,
      )
    )
  return out


{symbol: await open_orders(symbol) for symbol in MARKETS['spot']}

{'BTC-USDT': [], 'ETH-USDT': [], 'KCS-USDT': []}

In [6]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  page = await client.spot.orders_hf.get_trade_history(
    symbol=symbol, start_at=start, end_at=end
  )
  out: list[Trade] = []
  for f in page['items']:
    size = Decimal(f['size'])
    fee_amount = Decimal(f['fee'])
    out.append(
      Trade(
        id=str(f['tradeId']),
        price=Decimal(f['price']),
        qty=size if f['side'] == 'buy' else -size,
        time=f['createdAt'],
        maker=f['liquidity'] == 'maker',
        fee=Trade.Fee(amount=fee_amount, asset=f['feeCurrency'])
        if fee_amount
        else None,
        details=f,
      )
    )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in MARKETS['spot']}

{'BTC-USDT': [], 'ETH-USDT': [], 'KCS-USDT': []}

In [7]:
def trades_stream(symbol: str):
  # Spot/Margin's private order-lifecycle feed is account-wide (one topic covers every
  # symbol), so filter to `symbol` and to fill (`match`) events client-side.
  def parse(event: OrderChangeEvent) -> Trade | None:
    if event.get('type') != 'match' or event.get('symbol') != symbol:
      return None
    match_size = event.get('matchSize')
    match_price = event.get('matchPrice')
    if match_size is None or match_price is None:
      return None
    return Trade(
      id=event.get('tradeId'),
      price=match_price,
      qty=match_size if event['side'] == 'buy' else -match_size,
      time=event['orderTime'],
      maker=event.get('liquidity') == 'maker',
      fee=None,
      details=event,
    )

  return (
    client.streams.spot_margin_private.order_v2()
    .map(parse)
    .filter(lambda t: t is not None)
  )


async with trades_stream(MARKETS['spot'][0]) as stream:
  it = aiter(stream)
  try:
    result = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    result = (
      'no new trades observed in 5s (expected -- no live trading on this account)'
    )
result

'no new trades observed in 5s (expected -- no live trading on this account)'

In [8]:
async def position(symbol: str) -> Position:
  sym = await client.spot.symbol(symbol)
  accounts = await client.account.spot_accounts(
    currency=sym['baseCurrency'], type='trade'
  )
  size = sum((Decimal(a['balance']) for a in accounts), Decimal(0))
  return Position(size=size)


{symbol: await position(symbol) for symbol in MARKETS['spot']}

{'BTC-USDT': Position(size=Decimal('0')),
 'ETH-USDT': Position(size=Decimal('0')),
 'KCS-USDT': Position(size=Decimal('0'))}

In [9]:
async def collateral(symbol: str) -> Collateral:
  sym = await client.spot.symbol(symbol)
  accounts = await client.account.spot_accounts(
    currency=sym['quoteCurrency'], type='trade'
  )
  balance = accounts[0] if accounts else None
  equity = Decimal(balance['balance']) if balance else Decimal(0)
  free = Decimal(balance['available']) if balance else Decimal(0)
  return Collateral(equity=equity, free_collateral=free)


{symbol: await collateral(symbol) for symbol in MARKETS['spot']}

{'BTC-USDT': Collateral(equity=Decimal('0.39968083'), free_collateral=Decimal('0.39968083')),
 'ETH-USDT': Collateral(equity=Decimal('0.39968083'), free_collateral=Decimal('0.39968083')),
 'KCS-USDT': Collateral(equity=Decimal('0.39968083'), free_collateral=Decimal('0.39968083'))}

In [10]:
async def available_notional(symbol: str) -> Decimal:
  c = await collateral(symbol)
  return c.free_collateral


{symbol: await available_notional(symbol) for symbol in MARKETS['spot']}

{'BTC-USDT': Decimal('0.39968083'),
 'ETH-USDT': Decimal('0.39968083'),
 'KCS-USDT': Decimal('0.39968083')}

In [ ]:
async def place_order(
  symbol: str, order: Order, *, settings: Settings = {}
) -> OrderResponse:
  qty = Decimal(order['qty'])
  side: Literal['buy', 'sell'] = 'buy' if qty > 0 else 'sell'
  size = abs(qty)
  body: HfAddOrderLimit | HfAddOrderMarket
  if order['type'] == 'MARKET':
    body = {'symbol': symbol, 'type': 'market', 'side': side, 'size': size}
  else:
    limit: HfAddOrderLimit = {
      'symbol': symbol,
      'type': 'limit',
      'side': side,
      'price': Decimal(order['price']),
      'size': size,
    }
    if order['type'] == 'POST_ONLY':
      limit['postOnly'] = True
    body = limit
  raw = await client.spot.orders_hf.add(body)
  return OrderResponse(id=raw['orderId'], details=raw)


# Not executed here -- would place a real order on the account.
await place_order(
  'BTC-USDT', {'qty': Decimal('0.0001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

In [ ]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.spot.orders_hf.cancel_by_order_id(order_id=id, symbol=symbol)


# Not executed here -- would cancel a real order on the account.
await cancel_order('BTC-USDT', '123456')

## `PerpMarket` (Futures)

KuCoin futures denominate order/position/book quantities in **lots (contracts)**, not
base-asset units -- converting to base units needs each contract's `multiplier`
(`futures.symbol`), and is negative/inverted for inverse contracts. This notebook passes
the raw lot count through as `qty`/`size` (documented in the coverage note below) rather
than silently mis-converting.

In [11]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  size = '100' if levels and levels > 20 else '20'
  raw = await client.futures.part_orderbook(size, symbol=symbol)
  book = Book(
    bids=[Book.Entry(Decimal(str(p)), Decimal(q)) for p, q in raw['bids']],
    asks=[Book.Entry(Decimal(str(p)), Decimal(q)) for p, q in raw['asks']],
  )
  return book.limit(levels) if levels else book


{symbol: await depth(symbol, levels=5) for symbol in MARKETS['perp']}

{'XBTUSDTM': Book(bids=[Book.Entry(price=Decimal('79654.2'), qty=Decimal('568')), Book.Entry(price=Decimal('79651.2'), qty=Decimal('206')), Book.Entry(price=Decimal('79651.1'), qty=Decimal('80')), Book.Entry(price=Decimal('79651.0'), qty=Decimal('32')), Book.Entry(price=Decimal('79650.8'), qty=Decimal('75'))], asks=[Book.Entry(price=Decimal('79654.3'), qty=Decimal('175')), Book.Entry(price=Decimal('79657.4'), qty=Decimal('29')), Book.Entry(price=Decimal('79657.5'), qty=Decimal('92')), Book.Entry(price=Decimal('79658.6'), qty=Decimal('89')), Book.Entry(price=Decimal('79660.2'), qty=Decimal('4'))]),
 'ETHUSDTM': Book(bids=[Book.Entry(price=Decimal('2457.12'), qty=Decimal('2131')), Book.Entry(price=Decimal('2457.09'), qty=Decimal('21')), Book.Entry(price=Decimal('2457.04'), qty=Decimal('138')), Book.Entry(price=Decimal('2457.03'), qty=Decimal('141')), Book.Entry(price=Decimal('2457.01'), qty=Decimal('155'))], asks=[Book.Entry(price=Decimal('2457.13'), qty=Decimal('71')), Book.Entry(price=

In [12]:
def depth_stream(symbol: str):
  def to_book(update: FuturesOrderBookLevelUpdate) -> Book:
    return Book(
      bids=[Book.Entry(p, Decimal(q)) for p, q in update['bids']],
      asks=[Book.Entry(p, Decimal(q)) for p, q in update['asks']],
    )

  return client.streams.futures_public.orderbook_level5(symbol).map(to_book)


books: list[Book] = []
async with depth_stream(MARKETS['perp'][0]) as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('79654.2'), qty=Decimal('569')), Book.Entry(price=Decimal('79651.2'), qty=Decimal('206')), Book.Entry(price=Decimal('79651.1'), qty=Decimal('80')), Book.Entry(price=Decimal('79651'), qty=Decimal('32')), Book.Entry(price=Decimal('79650.8'), qty=Decimal('75'))], asks=[Book.Entry(price=Decimal('79654.3'), qty=Decimal('117')), Book.Entry(price=Decimal('79657.4'), qty=Decimal('29')), Book.Entry(price=Decimal('79657.5'), qty=Decimal('82')), Book.Entry(price=Decimal('79658.6'), qty=Decimal('89')), Book.Entry(price=Decimal('79660.2'), qty=Decimal('4'))]),
 Book(bids=[Book.Entry(price=Decimal('79654.2'), qty=Decimal('569')), Book.Entry(price=Decimal('79651.2'), qty=Decimal('206')), Book.Entry(price=Decimal('79651.1'), qty=Decimal('80')), Book.Entry(price=Decimal('79651'), qty=Decimal('32')), Book.Entry(price=Decimal('79650.8'), qty=Decimal('75'))], asks=[Book.Entry(price=Decimal('79654.3'), qty=Decimal('117')), Book.Entry(price=Decimal('79657.4'), qty=Decima

In [13]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  sym = await client.futures.symbol(symbol)
  return Rules(
    base=sym['baseCurrency'],
    quote=sym['quoteCurrency'],
    fee_asset=sym['settleCurrency'],
    tick_size=Decimal(str(sym['tickSize'])),
    step_size=Decimal(sym['lotSize']),
    fixed_min_qty=Decimal(sym['lotSize']),
    max_qty=Decimal(sym['maxOrderQty']),
    maker_fee=Decimal(str(sym['makerFeeRate'])),
    taker_fee=Decimal(str(sym['takerFeeRate'])),
    api=sym['status'] == 'Open',
    details=sym,
  )


{symbol: await rules(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': Rules(base='XBT', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('1'), fixed_min_qty=Decimal('1'), min_value=None, max_qty=Decimal('1000000'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0002'), taker_fee=Decimal('0.0006'), api=True, details={'symbol': 'XBTUSDTM', 'displaySymbol': 'XBTUSDTM', 'rootSymbol': 'USDT', 'type': 'FFWCSX', 'firstOpenDate': datetime.datetime(2020, 3, 30, 8, 0, tzinfo=datetime.timezone.utc), 'expireDate': None, 'settleDate': None, 'baseCurrency': 'XBT', 'displayBaseCurrency': 'XBT', 'quoteCurrency': 'USDT', 'settleCurrency': 'USDT', 'maxOrderQty': 1000000, 'marketMaxOrderQty': 1000000, 'maxPrice': 1000000.0, 'lotSize': 1, 'tickSize': 0.1, 'indexPriceTickSize': 0.01, 'multiplier': 0.001, 'initialMargin': 0.008, 'maintainMargin': 0.004, 'maxRiskLimit': 250000, 'minRiskLimit': 250000, 'riskStep': 125000, 'makerFeeRate': 0.0002, 'takerFeeRate': 0.0006, 'takerFixFee

In [14]:
async def open_orders(symbol: str) -> list[OrderState]:
  page = await client.futures.orders.get_order_list(status='active', symbol=symbol)
  out: list[OrderState] = []
  for o in page['items']:
    size = Decimal(o['size'])
    dealt = Decimal(o['dealSize'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(
      OrderState(
        id=o['id'],
        price=Decimal(o['price']),
        qty=sign * size,
        filled_qty=sign * dealt,
        active=o['isActive'],
        details=o,
      )
    )
  return out


{symbol: await open_orders(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': [], 'ETHUSDTM': [], 'KCSUSDTM': []}

In [15]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  # Each call spans at most 7 days -- the caller (or the SDK's own pagination layer, for
  # a wider window) is responsible for chunking a longer range into 7-day requests.
  page = await client.futures.orders.get_trade_history(
    symbol=symbol, start_at=start, end_at=end
  )
  out: list[Trade] = []
  for f in page['items']:
    size = Decimal(f['size'])
    fee = f['openFeePay'] + f['closeFeePay']
    out.append(
      Trade(
        id=f['tradeId'],
        price=f['price'],
        qty=size if f['side'] == 'buy' else -size,
        time=f['tradeTime'],
        maker=f['liquidity'] == 'maker',
        fee=Trade.Fee(amount=fee, asset=f['feeCurrency']) if fee else None,
        details=f,
      )
    )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in MARKETS['perp']}

{'XBTUSDTM': [], 'ETHUSDTM': [], 'KCSUSDTM': []}

In [16]:
def trades_stream(symbol: str):
  def parse(event: FuturesOrderEvent) -> Trade | None:
    if event.get('type') != 'match':
      return None
    match_size = event.get('matchSize')
    match_price = event.get('matchPrice')
    if match_size is None or match_price is None:
      return None
    size = Decimal(match_size)
    return Trade(
      id=event.get('tradeId'),
      price=match_price,
      qty=size if event['side'] == 'buy' else -size,
      time=event['orderTime'],
      maker=event.get('liquidity') == 'maker',
      fee=None,
      details=event,
    )

  return (
    client.streams.futures_private.order(symbol)
    .map(parse)
    .filter(lambda t: t is not None)
  )


async with trades_stream(MARKETS['perp'][0]) as stream:
  it = aiter(stream)
  try:
    result = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    result = (
      'no new trades observed in 5s (expected -- no live trading on this account)'
    )
result

'no new trades observed in 5s (expected -- no live trading on this account)'

In [17]:
async def perp_position(symbol: str) -> PerpPosition:
  row = await client.futures.positions.get_position_details(symbol=symbol)
  return PerpPosition(
    size=Decimal(row['currentQty']), entry_price=Decimal(str(row['avgEntryPrice']))
  )


{symbol: await perp_position(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0')),
 'ETHUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0')),
 'KCSUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0'))}

In [18]:
async def position(symbol: str) -> PerpPosition:
  return await perp_position(symbol)


{symbol: await position(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0')),
 'ETHUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0')),
 'KCSUSDTM': PerpPosition(size=Decimal('0'), entry_price=Decimal('0.0'))}

In [ ]:
async def perp_collateral(symbol: str) -> PerpCollateral:
  sym, position_row = await asyncio.gather(
    client.futures.symbol(symbol),
    client.futures.positions.get_position_details(symbol=symbol),
  )
  account = await client.account.futures_account(currency=sym['settleCurrency'])
  equity = Decimal(str(account['accountEquity']))
  leverage = (
    Decimal(str(position_row['leverage'])) if position_row.get('isOpen') else Decimal(0)
  )
  margin_mode = 'isolated' if position_row.get('marginMode') == 'ISOLATED' else 'cross'
  return PerpCollateral(
    equity=equity,
    free_collateral=Decimal(str(account['availableBalance'])),
    initial_margin=Decimal(str(account['positionMargin'] + account['orderMargin'])),
    maintenance_margin=Decimal(str(position_row.get('maintMargin') or 0)),
    leverage=leverage,
    margin_mode=margin_mode,
  )


# not executed: this API key lacks the `Futures` permission `GET /api/v1/account-overview` requires (live 404; see the coverage note), so `account.futures_account` cannot be exercised
{symbol: await perp_collateral(symbol) for symbol in MARKETS['perp']}

In [ ]:
async def collateral(symbol: str) -> PerpCollateral:
  return await perp_collateral(symbol)


# not executed: this API key lacks the `Futures` permission `GET /api/v1/account-overview` requires (live 404; see the coverage note), so `account.futures_account` cannot be exercised
{symbol: await collateral(symbol) for symbol in MARKETS['perp']}

In [ ]:
async def available_notional(symbol: str) -> Decimal:
  c, sym = await asyncio.gather(perp_collateral(symbol), client.futures.symbol(symbol))
  return c.free_collateral * Decimal(sym['maxLeverage'])


# not executed: this API key lacks the `Futures` permission `GET /api/v1/account-overview` requires (live 404; see the coverage note), so `account.futures_account` cannot be exercised
{symbol: await available_notional(symbol) for symbol in MARKETS['perp']}

In [ ]:
async def place_order(
  symbol: str, order: Order, *, settings: Settings = {}
) -> OrderResponse:
  import uuid

  qty = Decimal(order['qty'])
  side: Literal['buy', 'sell'] = 'buy' if qty > 0 else 'sell'
  size = int(abs(qty))
  client_oid = str(uuid.uuid4())
  body: FuturesAddOrderLimit | FuturesAddOrderMarket
  if order['type'] == 'MARKET':
    body = {
      'clientOid': client_oid,
      'symbol': symbol,
      'side': side,
      'leverage': 1,
      'size': size,
      'type': 'market',
    }
  else:
    limit: FuturesAddOrderLimit = {
      'clientOid': client_oid,
      'symbol': symbol,
      'side': side,
      'leverage': 1,
      'size': size,
      'type': 'limit',
      'price': Decimal(order['price']),
    }
    if order['type'] == 'POST_ONLY':
      limit['postOnly'] = True
    body = limit
  raw = await client.futures.orders.add(body)
  return OrderResponse(id=raw['orderId'], details=raw)


# Not executed here -- would place a real order on the account.
await place_order(
  'XBTUSDTM', {'qty': Decimal('1'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

In [ ]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.futures.orders.cancel_by_order_id(id)


# Not executed here -- would cancel a real order on the account.
await cancel_order('XBTUSDTM', '123456')

In [22]:
async def index(symbol: str, *, settings: Settings = {}) -> Decimal:
  sym = await client.futures.symbol(symbol)
  return Decimal(str(sym['indexPrice']))


{symbol: await index(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': Decimal('79699.2'),
 'ETHUSDTM': Decimal('2458.48'),
 'KCSUSDTM': Decimal('7.097')}

In [23]:
async def next_funding(symbol: str) -> NextFunding:
  sym = await client.futures.symbol(symbol)
  # `nextFundingRateDateTime`/`fundingRateGranularity` are `None` on a dated (non-perpetual)
  # contract, which has no funding mechanism -- every `MARKETS['perp']` symbol here is a
  # perpetual, so this should never actually trip.
  next_funding_time = sym['nextFundingRateDateTime']
  granularity = sym['fundingRateGranularity']
  assert next_funding_time is not None and granularity is not None, (
    f'{symbol} has no funding schedule -- not a perpetual contract'
  )
  return NextFunding(
    rate=Decimal(str(sym['fundingFeeRate'])),
    time=next_funding_time,
    interval=timedelta(milliseconds=granularity),
  )


{symbol: await next_funding(symbol) for symbol in MARKETS['perp']}

{'XBTUSDTM': NextFunding(rate=Decimal('-0.000021'), time=datetime.datetime(2026, 9, 5, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'ETHUSDTM': NextFunding(rate=Decimal('0.00002'), time=datetime.datetime(2026, 9, 5, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'KCSUSDTM': NextFunding(rate=Decimal('0.001323'), time=datetime.datetime(2026, 9, 5, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800))}

In [24]:
async def funding_rates(
  symbol: str,
  start: datetime | None = None,
  end: datetime | None = None,
) -> list[FundingRate]:
  # `public_funding_history` requires both `from_`/`to` -- there's no venue call that
  # actually returns "everything since the earliest available" in one shot, so `start=None`
  # here falls back to a 7-day window rather than the abstract docstring's literal contract.
  end = end or datetime.now(timezone.utc)
  start = start or end - timedelta(days=7)
  raw = await client.futures.funding_fees.public_funding_history(
    symbol=symbol, from_=start, to=end
  )
  return [
    FundingRate(rate=Decimal(str(r['fundingRate'])), time=r['timepoint']) for r in raw
  ]


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
{symbol: await funding_rates(symbol, start, end) for symbol in MARKETS['perp']}

{'XBTUSDTM': [FundingRate(rate=Decimal('0.000004'), time=datetime.datetime(2026, 9, 5, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('-0.000037'), time=datetime.datetime(2026, 9, 5, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000051'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000044'), time=datetime.datetime(2026, 9, 4, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000036'), time=datetime.datetime(2026, 9, 4, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000044'), time=datetime.datetime(2026, 9, 3, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000074'), time=datetime.datetime(2026, 9, 3, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000049'), time=datetime.datetime(2026, 9, 3, 0, 0, tzinfo=datetime

In [25]:
async def funding_payments(
  symbol: str, start: datetime, end: datetime
) -> list[FundingPayment]:
  page = await client.futures.funding_fees.private_funding_history(
    symbol=symbol, start_at=start, end_at=end
  )
  # KuCoin's `funding` is positive when *received*; the SDK's `FundingPayment.amount` is
  # positive when *paid* -- opposite sign conventions, so this flips it.
  return [
    FundingPayment(amount=-Decimal(str(f['funding'])), time=f['timePoint'])
    for f in page['dataList']
  ]


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
{symbol: await funding_payments(symbol, start, end) for symbol in MARKETS['perp']}

{'XBTUSDTM': [], 'ETHUSDTM': [], 'KCSUSDTM': []}

In [26]:
# Diagnostic: is `account.futures_account`'s 404 caused by an unactivated futures
# wallet, or by the API key itself lacking the `Futures` permission scope?
# `account.api_key_info` (`GET /api/v1/user/api-key`) only echoes metadata about the
# key signing the request -- read-only, safe to call live.
await client.account.api_key_info()

{'remark': 'Typed Dev',
 'apiKey': '6a76104c0ed31a0001b61cf3',
 'apiVersion': 3,
 'permission': 'General,Unified,Spot,Earn,InnerTransfer,Margin',
 'ipWhitelist': '18.**.118',
 'createdAt': datetime.datetime(2026, 8, 7, 17, 5, 16, tzinfo=datetime.timezone.utc),
 'uid': 232956088,
 'isMaster': True,
 'region': 'ES',
 'kycStatus': 1,
 'siteType': 'global'}

### Coverage assessment: `Market` / `PerpMarket`

**Spot: fully supported**, live-tested for depth/streaming depth/rules/open
orders/trades history/streaming trades/position/collateral/available notional above.
`orders_hf` (the high-frequency order book) is the natural mapping for `place_order`/
`cancel_order`/`open_orders`/`trades_history` -- KuCoin's plain (non-hf) order book is a
legacy surface the docs steer new integrations away from. `trades_stream` has no
per-symbol private trade feed; it's built by filtering the account-wide `order_v2`
lifecycle feed down to `type == 'match'` events for the requested symbol.

**Futures: fully supported for read/write shape**, with one structural mismatch worth
flagging: KuCoin futures quantities (`Book` levels, `OrderState.qty`, `PerpPosition.size`)
are all in **lots (contracts)**, not base-asset units the way `Book`/`Position`'s
docstrings imply -- converting requires each contract's `multiplier` (and sign, for an
inverse contract), which this notebook does not attempt (it passes the lot count through
as-is). `trades_history`/`funding_payments`/`get_trade_history` are further capped to a
7-day window per call (documented on `futures.orders.get_trade_history`); a full
`PaginatedResponse` implementation would need to chunk a wider `start`/`end` into 7-day
slices itself, since KuCoin's own pagination only walks pages *within* one window.
`perp_collateral`'s `initial_margin`/`maintenance_margin` are approximated from
`account.futures_account` (account-level `positionMargin + orderMargin`) and the single
requested position's `maintMargin` respectively -- KuCoin has no one call that reports
both figures already aggregated the way the SDK's dataclass expects for a multi-position
account. On *this* account `account.futures_account` (`GET /api/v1/account-overview`)
returns a live `404` regardless of currency, while `futures.positions.get_position_*`
endpoints succeed with zeroed rows. The diagnostic cell above calls the read-only
`account.api_key_info` (`GET /api/v1/user/api-key`) to check why: this key's granted
`permission` scopes are `General,Unified,Spot,Earn,InnerTransfer,Margin` -- no
`Futures`. KuCoin's docs list `Futures` as the required permission for [`Get Account -
Futures`](https://www.kucoin.com/docs-new/rest/account-info/account-funding/get-account-futures),
and only `General` for [`Get Position
List`](https://www.kucoin.com/docs-new/rest/futures-trading/positions/get-position-list),
which lines up exactly with what 404s and what succeeds here. So this looks like a
permission-scope gap on the API key itself -- not, as first guessed, an unactivated or
unfunded futures wallet -- though KuCoin's own [error-code
reference](https://www.kucoin.com/docs-new/error-code/futures) documents a missing scope
as `400007` `Access Denied`, not a bare `404`, so the exact shape of this response isn't
itself explained by the docs. Confirming this would mean regenerating the key with
`Futures` enabled, which this notebook does not attempt since it would mutate account/key
state. Real errors from the gap propagate untouched through `perp_collateral`/`collateral`/
`available_notional` below rather than being papered over.